# Peaked-circuit MPS smoke test

Run the deterministic mirrored peaked family through Qiskit Aer MPS and MettleQ routed MPS with the same shots and known mode.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
from qiskit_aer import AerSimulator
from mettleq.peaked import build_mirrored_peaked_circuit

circuit = build_mirrored_peaked_circuit(8, depth=2, topology="long_range", seed=153, measure=True)
peak = circuit.metadata["peak_bitstring"]
expected_probability = circuit.metadata["expected_peak_probability"]
shots = 4096
aer = AerSimulator(method="matrix_product_state")
aer_circuit = transpile(circuit, aer, optimization_level=1)

def run_reference():
    return aer.run(aer_circuit, shots=shots, seed_simulator=153).result().get_counts()

reference, reference_ms, _ = benchmark(run_reference)
backend = MettleQBackend(
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
)
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    return backend.run(compiled, shots=shots, seed_simulator=153).result().get_counts()

candidate, mettleq_ms, _ = benchmark(run_mettleq)
tvd = total_variation_distance(reference, candidate)
observed = candidate.get(peak, 0) / shots
mode = max(candidate, key=candidate.get)
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/16_peaked_circuit_smoke.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="expected mode, peak probability atol=0.025, and TVD<=0.06",
    passed=mode == peak and abs(observed - expected_probability) <= 0.025 and tvd <= 0.06,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"tvd": tvd, "expected_peak": peak, "mettleq_mode": mode, "expected_probability": expected_probability, "observed_probability": observed},
    notes="This mirrored regression is not the full 56-qubit P9 quantum-advantage instance.",
)

TUTORIAL_RESULT::{"check": "expected mode, peak probability atol=0.025, and TVD<=0.06", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"expected_peak": "01001100", "expected_probability": 0.9500366589899866, "mettleq_mode": "01001100", "observed_probability": 0.955078125, "tvd": 0.0048828125}, "mettleq_median_ms": 29.768083011731505, "notebook": "qiskit/16_peaked_circuit_smoke.ipynb", "notes": "This mirrored regression is not the full 56-qubit P9 quantum-advantage instance.", "passed": true, "python": "3.13.2", "reference_median_ms": 5.589250009506941, "reference_over_mettleq": 0.18775982340899264, "schema_version": 1, "selected_device": "cpu", "selected_method": "matrix_product_state"}
